# Klasifikasi Survival Penumpang Titanic Menggunakan ResNet50 Berbasis TensorFlow dengan Optimisasi Gauss–Newton


# 0) Import Library

In [ ]:
import os
import re
import math
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# 1) Paths

In [17]:
TRAIN_PATH = "/kaggle/input/competitions/titanic/train.csv"
TEST_PATH = "/kaggle/input/competitions/titanic/test.csv"

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "train.csv"
if not os.path.exists(TEST_PATH):
    TEST_PATH = "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)


Train shape: (891, 12)
Test shape : (418, 11)


# 2) Feature engineering

In [18]:
def extract_title(name: str) -> str:
    match = re.search(r",\s*([^\.]+)\.", str(name))
    return match.group(1).strip() if match else "Unknown"

def build_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_y = train_df["Survived"].copy()
    test_passenger_id = test_df["PassengerId"].copy()

    full = pd.concat(
        [train_df.drop(columns=["Survived"]).copy(), test_df.copy()],
        axis=0,
        ignore_index=True
    )

    full["Title"] = full["Name"].apply(extract_title)
    full["Title"] = full["Title"].replace({
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
        "Lady": "Rare", "Countess": "Rare", "Capt": "Rare",
        "Col": "Rare", "Don": "Rare", "Dr": "Rare", "Major": "Rare",
        "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"
    })

    full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
    full["IsAlone"] = (full["FamilySize"] == 1).astype(int)
    full["CabinKnown"] = full["Cabin"].notna().astype(int)
    full["CabinDeck"] = full["Cabin"].fillna("U").astype(str).str[0]

    deck_counts = full["CabinDeck"].value_counts()
    rare_decks = deck_counts[deck_counts < 10].index
    full.loc[full["CabinDeck"].isin(rare_decks), "CabinDeck"] = "Rare"

    ticket_counts = full["Ticket"].value_counts()
    full["TicketGroupSize"] = full["Ticket"].map(ticket_counts).fillna(1).astype(int)

    full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
    full["Fare"] = full.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))

    age_guess = full.groupby(["Sex", "Pclass", "Title"])["Age"].transform("median")
    full["Age"] = full["Age"].fillna(age_guess)
    full["Age"] = full["Age"].fillna(full["Age"].median())

    full["LogFare"] = np.log1p(full["Fare"])
    full["FarePerPerson"] = full["Fare"] / full["FamilySize"].replace(0, 1)
    full["AgeClass"] = full["Age"] * full["Pclass"]
    full["Child"] = (full["Age"] < 16).astype(int)

    full["Sex"] = full["Sex"].map({"male": 0, "female": 1}).astype(int)

    drop_cols = ["Name", "Ticket", "Cabin"]
    full = full.drop(columns=drop_cols)

    full = pd.get_dummies(
        full,
        columns=["Embarked", "Title", "CabinDeck", "Pclass"],
        drop_first=False
    )

    X_train = full.iloc[: len(train_df)].copy()
    X_test = full.iloc[len(train_df):].copy()

    X_train = X_train.drop(columns=["PassengerId"])
    X_test = X_test.drop(columns=["PassengerId"])

    return X_train, train_y.astype(np.float32).values, X_test, test_passenger_id.values

X_train_df, y, X_test_df, test_passenger_ids = build_features(train_df, test_df)

print("Feature count:", X_train_df.shape[1])

Feature count: 33


# 3) Dual preprocessing

In [19]:
tab_scaler = StandardScaler()
img_scaler = MinMaxScaler(feature_range=(0.0, 255.0))

X_train_tab_all = tab_scaler.fit_transform(X_train_df).astype(np.float32)
X_test_tab = tab_scaler.transform(X_test_df).astype(np.float32)

X_train_img_base = img_scaler.fit_transform(X_train_df).astype(np.float32)
X_test_img_base = img_scaler.transform(X_test_df).astype(np.float32)

def tabular_to_pseudo_image(X, image_size=(32, 32), channels=3):
    h, w = image_size
    total = h * w * channels
    X = np.asarray(X, dtype=np.float32)
    repeats = math.ceil(total / X.shape[1])
    tiled = np.tile(X, (1, repeats))[:, :total]
    images = tiled.reshape(-1, h, w, channels)
    return images.astype(np.float32)

X_train_img_all = tabular_to_pseudo_image(X_train_img_base)
X_test_img = tabular_to_pseudo_image(X_test_img_base)

X_train_tab, X_val_tab, X_train_img, X_val_img, y_train, y_val = train_test_split(
    X_train_tab_all,
    X_train_img_all,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train split:", X_train_tab.shape, X_train_img.shape)
print("Val split  :", X_val_tab.shape, X_val_img.shape)


Train split: (712, 33) (712, 32, 32, 3)
Val split  : (179, 33) (179, 32, 32, 3)


# 4) ResNet50 + tabular hybrid model

In [ ]:
def build_model(tab_dim, image_shape=(32, 32, 3), use_imagenet=False):
    weights = "imagenet" if use_imagenet else None

    try:
        resnet_base = ResNet50(
            include_top=False,
            weights=weights,
            input_shape=image_shape,
            pooling="avg"
        )
        print(f"ResNet50 loaded with weights={weights}")
    except Exception as e:
        print("ImageNet weights gagal di-load, fallback ke weights=None")
        print("Detail:", e)
        resnet_base = ResNet50(
            include_top=False,
            weights=None,
            input_shape=image_shape,
            pooling="avg"
        )
        use_imagenet = False

    resnet_base.trainable = not use_imagenet

    img_input = layers.Input(shape=image_shape, name="img_input")
    tab_input = layers.Input(shape=(tab_dim,), name="tab_input")

    x_img = preprocess_input(img_input)
    x_img = resnet_base(x_img)
    x_img = layers.Dense(64, activation="relu")(x_img)
    x_img = layers.Dropout(0.25)(x_img)

    x_tab = layers.Dense(64, activation="relu")(tab_input)
    x_tab = layers.BatchNormalization()(x_tab)
    x_tab = layers.Dropout(0.20)(x_tab)
    x_tab = layers.Dense(32, activation="relu")(x_tab)

    x = layers.Concatenate()([x_img, x_tab])
    embedding = layers.Dense(64, activation="relu", name="embedding")(x)
    output = layers.Dense(1, activation="sigmoid", name="classifier")(embedding)

    model = Model(inputs=[img_input, tab_input], outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4 if not use_imagenet else 3e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    embedding_model = Model(inputs=model.inputs, outputs=model.get_layer("embedding").output)
    return model, embedding_model

USE_IMAGENET = False

model, embedding_model = build_model(tab_dim=X_train_tab.shape[1], use_imagenet=USE_IMAGENET)
model.summary()

early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=4,
    mode="max",
    restore_best_weights=True
)

history = model.fit(
    [X_train_img, X_train_tab],
    y_train,
    validation_data=([X_val_img, X_val_tab], y_val),
    epochs=12,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

val_probs_base = model.predict([X_val_img, X_val_tab], verbose=0).ravel()
val_pred_base = (val_probs_base >= 0.5).astype(int)
val_acc_base = accuracy_score(y_val.astype(int), val_pred_base.astype(int))
print(f"Validation accuracy (Adam/base head): {val_acc_base:.4f}")

2026-04-08 10:09:07.122102: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ResNet50 loaded with weights=None


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_input           │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 32, 32)    │          0 │ img_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 32, 32)    │          0 │ img_input[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 32, 32)    │          0 │ img_input[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 32, 32, 3) │          0 │ get_item[0][0],   │
│                     │                   │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tab_input           │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32, 3) │          0 │ stack[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      2,176 │ tab_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 2048)      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64)        │        256 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │    131,136 │ resnet50[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding (Dense)   │ (None, 64)        │      6,208 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier (Dense)  │ (None, 1)         │         65 │ embedding[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,729,633 (90.52 MB)

 Trainable params: 23,676,385 (90.32 MB)

 Non-trainable params: 53,248 (208.00 KB)

Epoch 1/12


# 5) Damped Gauss-Newton optimizer for the final logistic head

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def fit_damped_gauss_newton(X_embed, y, damping=1e-2, max_iter=25, tol=1e-6):
    X_embed = np.asarray(X_embed, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1, 1)

    Xb = np.concatenate([X_embed, np.ones((X_embed.shape[0], 1))], axis=1)
    n_features = Xb.shape[1]

    w = np.zeros((n_features, 1), dtype=np.float64)
    prev_loss = np.inf

    for it in range(max_iter):
        z = Xb @ w
        p = sigmoid(z)
        residual = p - y

        weights_diag = (p * (1.0 - p)).ravel() + 1e-8
        XT_W = Xb.T * weights_diag
        H = XT_W @ Xb + damping * np.eye(n_features)
        g = Xb.T @ residual

        step = np.linalg.solve(H, g)
        w -= step

        eps = 1e-9
        loss = -np.mean(y * np.log(p + eps) + (1.0 - y) * np.log(1.0 - p + eps))
        print(f"GN iter {it+1:02d} | loss={float(loss):.6f} | step_norm={float(np.linalg.norm(step)):.6f}")

        if abs(prev_loss - loss) < tol:
            break
        prev_loss = loss

    return w

def predict_proba_gn(X_embed, params):
    X_embed = np.asarray(X_embed, dtype=np.float64)
    Xb = np.concatenate([X_embed, np.ones((X_embed.shape[0], 1))], axis=1)
    return sigmoid(Xb @ params).ravel()

H_train = embedding_model.predict([X_train_img, X_train_tab], verbose=0)
H_val = embedding_model.predict([X_val_img, X_val_tab], verbose=0)
H_test = embedding_model.predict([X_test_img, X_test_tab], verbose=0)

gn_params = fit_damped_gauss_newton(H_train, y_train, damping=1e-1, max_iter=20)

val_probs_gn = predict_proba_gn(H_val, gn_params)

best_threshold = 0.5
best_acc_gn = 0.0
for t in np.linspace(0.30, 0.70, 41):
    pred_t = (val_probs_gn >= t).astype(int)
    acc_t = accuracy_score(y_val.astype(int), pred_t)
    if acc_t > best_acc_gn:
        best_acc_gn = acc_t
        best_threshold = float(t)

print(f"Validation accuracy (Gauss-Newton head): {best_acc_gn:.4f} at threshold={best_threshold:.2f}")

use_gn_head = best_acc_gn >= val_acc_base
print("Use Gauss-Newton head for test prediction:", use_gn_head)

if use_gn_head:
    test_probs = predict_proba_gn(H_test, gn_params)
    test_pred = (test_probs >= best_threshold).astype(int)
else:
    test_probs = model.predict([X_test_img, X_test_tab], verbose=0).ravel()
    test_pred = (test_probs >= 0.5).astype(int)


# 6) Visualisasi Perbandingan: Adam vs Gauss-Newton

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, roc_auc_score
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = ["Adam\n(Tanpa Gauss-Newton)", "Damped Gauss-Newton"]
accuracies = [val_acc_base, best_acc_gn]
colors = ["#3498db", "#e74c3c"]

ax1 = axes[0, 0]
bars = ax1.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor="black", linewidth=1.5)
ax1.set_ylabel("Validation Accuracy", fontsize=11, fontweight="bold")
ax1.set_title("Perbandingan Akurasi Validasi", fontsize=12, fontweight="bold")
ax1.set_ylim([0.75, 0.85])
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, 
             f"{acc:.4f}", ha="center", va="bottom", fontweight="bold")

ax2 = axes[0, 1]
fpr_adam, tpr_adam, _ = roc_curve(y_val.astype(int), val_probs_base)
fpr_gn, tpr_gn, _ = roc_curve(y_val.astype(int), val_probs_gn)
auc_adam = auc(fpr_adam, tpr_adam)
auc_gn = auc(fpr_gn, tpr_gn)

ax2.plot(fpr_adam, tpr_adam, label=f"Adam (AUC={auc_adam:.4f})", linewidth=2.5, color="#3498db")
ax2.plot(fpr_gn, tpr_gn, label=f"Gauss-Newton (AUC={auc_gn:.4f})", linewidth=2.5, color="#e74c3c")
ax2.plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1)
ax2.set_xlabel("False Positive Rate", fontsize=11, fontweight="bold")
ax2.set_ylabel("True Positive Rate", fontsize=11, fontweight="bold")
ax2.set_title("Kurva ROC - Perbandingan Model", fontsize=12, fontweight="bold")
ax2.legend(loc="lower right", fontsize=10)
ax2.grid(alpha=0.3)

ax3 = axes[1, 0]
val_pred_adam = (val_probs_base >= 0.5).astype(int)
cm_adam = confusion_matrix(y_val.astype(int), val_pred_adam)
sns.heatmap(cm_adam, annot=True, fmt="d", cmap="Blues", ax=ax3, 
            cbar_kws={"label": "Jumlah"}, annot_kws={"fontsize": 12, "fontweight": "bold"})
ax3.set_title("Confusion Matrix - Adam (Threshold=0.5)", fontsize=12, fontweight="bold")
ax3.set_ylabel("Actual", fontsize=11, fontweight="bold")
ax3.set_xlabel("Predicted", fontsize=11, fontweight="bold")

ax4 = axes[1, 1]
val_pred_gn = (val_probs_gn >= best_threshold).astype(int)
cm_gn = confusion_matrix(y_val.astype(int), val_pred_gn)
sns.heatmap(cm_gn, annot=True, fmt="d", cmap="YlOrRd", ax=ax4, 
            cbar_kws={"label": "Jumlah"}, annot_kws={"fontsize": 12, "fontweight": "bold"})
ax4.set_title(f"Confusion Matrix - Gauss-Newton (Threshold={best_threshold:.2f})", 
              fontsize=12, fontweight="bold")
ax4.set_ylabel("Actual", fontsize=11, fontweight="bold")
ax4.set_xlabel("Predicted", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "comparison_adam_vs_gauss_newton.png"), dpi=150, bbox_inches="tight")
plt.show()

print("="*70)
print("RINGKASAN PERBANDINGAN")
print("="*70)
print(f"{'Metode':<25} {'Akurasi':<15} {'AUC-ROC':<15}")
print("-"*70)
print(f"{'Adam (Tanpa Gauss-Newton)':<25} {val_acc_base:<15.4f} {auc_adam:<15.4f}")
print(f"{'Damped Gauss-Newton':<25} {best_acc_gn:<15.4f} {auc_gn:<15.4f}")
print("="*70)
print(f"\nModel terbaik: {'Gauss-Newton' if use_gn_head else 'Adam'}")
print(f"Peningkatan akurasi: {(best_acc_gn - val_acc_base)*100:+.2f}%")


In [ ]:

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids.astype(int),
    "Survived": test_pred.astype(int)
})

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
output_path = os.path.join(output_dir, "submission.csv")
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(submission.head())

if output_dir != ".":
    submission.to_csv("submission.csv", index=False)
